# SpectraShift Week 7: aggregate foundation baselines
Use CPU with Internet off. Attach source v6, Week 6 complete, Week 7 contracts, pilots, probes, and all three Week 7 seed outputs.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week7.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 7 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week7-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 7 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

def install_offline_foundation_dependencies():
    wheels = sorted(INPUT.rglob('foundation-wheels')) + sorted(Path('/kaggle/working').rglob('foundation-wheels'))
    if not wheels: return
    missing = []
    for module, package in [('upath','universal-pathlib'), ('omegaconf','omegaconf'), ('iopath','iopath'), ('fvcore','fvcore'), ('einops','einops'), ('huggingface_hub','huggingface_hub')]:
        try: __import__(module)
        except ImportError: missing.append(package)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(wheels[0]), *missing])

WORK = Path('/kaggle/working/spectrashift-week7-complete')
WORK.mkdir(parents=True, exist_ok=True)
WEEK6_SUMMARY = unique_file('week6_run_summary.json')
CONTROLLED_CURVES = unique_file('controlled_curves.csv')
WEEK7_CONTRACTS = unique_file('week7_contracts_summary.json')
PILOTS = unique_file('week7_pilot_summary.json')
PROBES = unique_file('week7_probe_summary.json')
seed_paths = []
for seed in (17, 29, 43):
    matches = sorted(INPUT.rglob(f'week7_seed{seed}_summary.json'))
    assert len(matches) == 1, f'Expected one seed {seed} summary, found {matches}'
    seed_paths.append(matches[0])


In [ ]:
from spectrashift.train.week7 import aggregate_week7

summary = aggregate_week7(WEEK6_SUMMARY, CONTROLLED_CURVES, WEEK7_CONTRACTS, PILOTS, PROBES, seed_paths, WORK)
print(json.dumps({key: value for key, value in summary.items() if key not in {'foundation_runs','foundation_comparisons','week8_checkpoint_ledger'}}, indent=2))
assert summary['week7_complete'] and summary['week8_approved']
assert summary['foundation_full_run_count'] == 18
assert summary['foundation_linear_probe_count'] == 36
assert summary['foundation_knn_run_count'] == 6
assert summary['foundation_feature_cache_count'] == 2
assert summary['evaluation_labels_loaded'] is False
